# 1. Introduction and End-to-End Objective

This notebook plans an end-to-end workflow to evaluate, train, tune, and test machine learning models that predict 1-month stock direction labels (buy, hold, sell) from historical OHLCV-derived features.

Expected final outcome of implementation (in later coding step):
- Model comparison leaderboard
- Final hold-out test evaluation
- Strategy diagnostics (long/flat/short)
- Clear recommendation: which model to use, why, and with which exact parameters

## 1.1 Configuration Scope and Assumptions

Inputs expected at implementation time:
- Universe CSV path in data folder (NAME.csv)
- Matching history folder data/NAME/
- Label thresholds (+5% buy, -5% sell)
- Lookback window (last 6 years)
- Time-aware split ratios and quality filter thresholds

Planned logic:
- Validate all required paths and run settings.
- Record unresolved ambiguities (for example liquidity threshold defaults).
- Standardize run metadata for reproducibility.

Planned outputs:
- Configuration object
- Run metadata object
- Open questions log (if needed)

# 2. Data Ingestion and Universe Resolution

Inputs expected at implementation time:
- Configured universe CSV in data folder
- Matching history subfolder with one CSV per symbol

Planned logic:
- Load symbols from NAME.csv.
- Resolve data/NAME/ historical source path.
- Validate symbol list quality and per-symbol file availability.

Planned outputs:
- Clean symbol universe
- Resolved symbol-to-file map
- Ingestion diagnostics summary

## 2.1 Raw OHLCV Loading and Schema Validation

Inputs expected at implementation time:
- Clean symbol universe
- Symbol-to-file path map

Planned logic:
- Load one CSV per symbol.
- Enforce expected raw schema: date, open, high, low, close, adj_close, volume.
- Parse date consistently and standardize symbol column.
- Track missing files and schema mismatches without terminating the full run.

Planned outputs:
- Pooled raw multi-symbol DataFrame
- Symbol load status report

## 2.2 Six-Year Windowing and Data Integrity Checks

Inputs expected at implementation time:
- Pooled raw multi-symbol DataFrame

Planned logic:
- Derive start_date = latest available trading date minus 6 years.
- Filter rows to the rolling six-year horizon.
- Detect duplicates, negative/implausible values, and out-of-order dates.
- Summarize missingness by symbol and field.

Planned outputs:
- Windowed dataset
- Integrity diagnostics table

# 3. Cleaning and Universe Filtering

Inputs expected at implementation time:
- Windowed dataset and integrity diagnostics

Planned logic:
- Apply row-level cleaning policy (sorting, missing value handling, anomaly removal).
- Apply symbol-level filters: minimum history, minimum median volume, maximum missingness ratio.
- Keep exclusion reasons for auditability.

Planned outputs:
- Filtered modeling universe
- Cleaned pooled dataset
- Exclusion report

# 4. Feature Engineering (PRD 5.1 to 5.8)

Inputs expected at implementation time:
- Cleaned pooled dataset after universe filtering

Planned logic:
- Compute per-symbol feature groups exactly as specified in PRD sections 5.1 to 5.8:
  - 5.1 Base return and range features
  - 5.2 Multi-horizon return features
  - 5.3 Trend features (moving averages and relative position)
  - 5.4 Volatility features
  - 5.5 Volume and liquidity features
  - 5.6 Technical indicators (RSI, MACD, ATR)
  - 5.7 High/low context features
  - 5.8 Feature consolidation and sanity checks
- Enforce alignment by symbol and date, and drop rows lacking required lookback history.

Planned outputs:
- Feature-complete modeling dataset
- Feature quality diagnostics (distribution, correlation, near-constant columns)

# 5. Target Construction and Class Diagnostics

Inputs expected at implementation time:
- Feature-complete dataset with close prices

Planned logic:
- Compute forward 20-trading-day return aligned to features at time t.
- Construct label_1m with fixed thresholds:
  - buy = 1 if fwd_ret_20d > +5%
  - hold = 0 otherwise
  - sell = -1 if fwd_ret_20d < -5%
- Drop rows where forward return is undefined.
- Diagnose class balance overall and by year.

Planned outputs:
- Labeled dataset
- Class distribution diagnostics

# 6. Time-Aware Splitting and Preprocessing

Inputs expected at implementation time:
- Labeled dataset

Planned logic:
- Perform chronological splitting into train (~70%), validation (~15%), and test (~15%) with no shuffling.
- Record actual date cutoffs and sample sizes.
- Define leakage-safe preprocessing strategy:
  - Scaling for Logistic Regression, LinearSVC, MLP
  - No scaling required for tree-based models (RF, Extra Trees, HistGradientBoosting)
- Prepare model pipelines that fit transforms on training data only.

Planned outputs:
- Train/validation/test datasets
- Split report with cutoff dates
- Preprocessing pipeline definitions

# 7. Baseline and Candidate Model Training

Inputs expected at implementation time:
- Time-aware split datasets
- Preprocessing pipelines

Planned logic:
- Train baseline strategies (majority-class and/or always-hold baseline).
- Train candidate models from PRD:
  - LogisticRegression
  - RandomForestClassifier
  - ExtraTreesClassifier
  - HistGradientBoostingClassifier
  - LinearSVC
  - MLPClassifier
- Evaluate first-pass performance on validation data.

Planned outputs:
- Validation leaderboard with macro F1 as primary metric
- Class-wise metrics for buy/hold/sell
- Baseline comparison summary

## 7.1 Hyperparameter Tuning with TimeSeriesSplit

Inputs expected at implementation time:
- Validation leaderboard and shortlisted top models
- Training + validation data

Planned logic:
- Select top two candidate models by validation macro F1 and minority-class behavior.
- Use TimeSeriesSplit for chronology-respecting hyperparameter search.
- Track fold-level stability and training/runtime cost.

Planned outputs:
- Tuned model candidates with best parameter sets
- CV stability and efficiency summary

# 8. Final Test Evaluation and Strategy Diagnostics

Inputs expected at implementation time:
- Tuned finalist models
- Untouched hold-out test dataset

Planned logic:
- Refit finalists on training + validation period.
- Evaluate on hold-out test with:
  - Macro F1
  - Weighted F1
  - Balanced accuracy
  - Class-wise precision/recall/F1
  - Confusion matrix
- Analyze common misclassification patterns.
- Compute simple long/flat/short cumulative return diagnostic from predicted labels.

Planned outputs:
- Final test metrics report
- Confusion matrix outputs
- Strategy diagnostic outputs

# 9. Final Recommendation and Artifact Plan

Inputs expected at implementation time:
- Final metrics of evaluated models
- Strategy diagnostics and tuning summaries

Planned logic:
- Select one recommended model based on PRD priorities:
  - Macro F1 first
  - Buy/sell class quality
  - Stability across time-aware folds
  - Complexity/runtime trade-off
- State explicit recommendation output:
  - Recommended model name
  - Why it is preferred over runner-up
  - Exact parameters to use
  - Metric deltas versus runner-up
- Define artifact outputs for reproducibility.

Planned outputs:
- Recommendation block content specification
- Artifact list (leaderboard table, best-params JSON, final report)